# Egyptian Civil Law — RAG with Qwen3 + Neo4j

Ask any question in **English or Arabic**. The pipeline:

```
Question
   │
   ▼
① Metadata Extraction  (Qwen3:4b → JSON: keywords, topics, article numbers)
   │
   ▼
② Hybrid Retrieval from Neo4j
   ├── Keyword graph search   (Keyword nodes)
   ├── Section search         (Section hierarchy)
   ├── Article number lookup  (if numbers mentioned)
   └── Semantic vector search (BGE-M3 embedding → vector index)
   │
   ▼
③ Reranking & Deduplication  (combined score)
   │
   ▼
④ Answer Generation  (Qwen3:4b + top articles as context)
```

> **Note:** `qwen3.5` does not exist on Ollama — the correct model is `qwen3:4b`.

## Step 1 — Install Python Packages

In [ ]:
%%capture
!pip install neo4j FlagEmbedding transformers torch tqdm ollama

## Step 2 — Install & Start Ollama, Pull Qwen3:4b

In [ ]:
import subprocess, time, requests

# Install Ollama binary
print('Installing Ollama...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
               shell=True, capture_output=True)

# Start Ollama server in background
print('Starting Ollama server...')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until the server is ready
for _ in range(30):
    try:
        r = requests.get('http://localhost:11434')
        if r.status_code == 200:
            print('Ollama server is ready!')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start. Restart runtime and retry.')

In [ ]:
LLM_MODEL = 'qwen3:4b'

print(f'Pulling {LLM_MODEL}  (2.5 GB — takes ~3 min on Colab)...')
!ollama pull qwen3:4b
print('Model ready.')

## Step 3 — Neo4j Connection

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'neo4j+s://785ea338.databases.neo4j.io'
NEO4J_USER     = '785ea338'
NEO4J_PASSWORD = '2e3Vah_a8qA5Q14DaECSa87pj0LifbWK_sJq6kZWbsE'
NEO4J_DATABASE = '785ea338'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j Aura!')

## Step 4 — Load BGE-M3 Embedding Model

In [ ]:
from FlagEmbedding import BGEM3FlagModel

print('Loading BGE-M3  (first run downloads ~2 GB)...')
embed_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('BGE-M3 ready.')

def embed(text: str) -> list:
    """Return dense embedding vector for a single text string."""
    out = embed_model.encode([text], batch_size=1, max_length=512)
    return out['dense_vecs'][0].tolist()

## Step 5 — Metadata Extractor (Qwen3 → structured JSON)

The LLM reads the question and returns keywords, legal topics, and any article numbers mentioned.  
We use `/no_think` prefix so Qwen3 skips its reasoning chain for this quick extraction task.

In [ ]:
import ollama, json, re

EXTRACT_SYSTEM = """\
You are a metadata extractor for an Egyptian Civil Law database.
Given the user's question (English or Arabic), extract search metadata.

Return ONLY a valid JSON object — no markdown fences, no extra text:
{
  "keywords_en": ["list", "of", "english", "legal", "keywords"],
  "keywords_ar": ["قائمة", "الكلمات", "العربية"],
  "legal_topics": ["broad legal topics, e.g. prescription, contracts, property"],
  "article_numbers": [],
  "search_query": "one refined sentence capturing the core legal question"
}

Rules:
- article_numbers: list of integers if the question mentions specific articles, else []
- keywords_en / keywords_ar: 3-8 terms each, only what is clearly in the question
- legal_topics: 1-4 broad categories
- search_query: English even if the question is in Arabic
"""

def extract_metadata(question: str) -> dict:
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': EXTRACT_SYSTEM},
            {'role': 'user',   'content': '/no_think\n' + question},
        ],
        format='json',
        options={'temperature': 0.0},
    )
    raw = response['message']['content']
    # Strip any accidental <think>...</think> blocks
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: simple keyword split
        return {
            'keywords_en': question.split()[:5],
            'keywords_ar': [],
            'legal_topics': [],
            'article_numbers': [],
            'search_query': question,
        }

# Quick test
test_meta = extract_metadata('What happens when a law is repealed?')
print(json.dumps(test_meta, ensure_ascii=False, indent=2))

## Step 6 — Hybrid Retrieval from Neo4j

Three parallel signals — results are merged and deduplicated.

In [ ]:
# ── helpers ────────────────────────────────────────────────────────────────

def _run(q, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(q, **params).data()

# ── 1. Article number lookup ───────────────────────────────────────────────
def fetch_by_number(numbers: list) -> list:
    if not numbers:
        return []
    rows = _run(
        'MATCH (a:Article) WHERE a.number IN $nums '
        'RETURN a.id AS id, a.number AS number, '
        '       a.english AS english, a.arabic AS arabic',
        nums=numbers
    )
    return [{'id': r['id'], 'number': r['number'],
             'english': r['english'], 'arabic': r['arabic'],
             'kw_score': 1.0, 'sem_score': 1.0, 'source': 'direct'}
            for r in rows]

# ── 2. Keyword + Section graph search ─────────────────────────────────────
KEYWORD_Q = """
UNWIND $keywords AS kw
MATCH (a:Article)-[:HAS_KEYWORD]->(k:Keyword)
WHERE toLower(k.name) CONTAINS toLower(kw)
WITH  a, count(DISTINCT k) AS hits
ORDER BY hits DESC
LIMIT  $limit
RETURN a.id AS id, a.number AS number,
       a.english AS english, a.arabic AS arabic,
       hits AS kw_hits
"""

SECTION_Q = """
UNWIND $topics AS topic
MATCH (s:Section)<-[:IN_SECTION]-(a:Article)
WHERE toLower(s.name) CONTAINS toLower(topic)
WITH  a, count(DISTINCT s) AS hits
ORDER BY hits DESC
LIMIT  $limit
RETURN a.id AS id, a.number AS number,
       a.english AS english, a.arabic AS arabic,
       hits AS kw_hits
"""

def fetch_by_keywords(keywords_en: list, keywords_ar: list,
                      legal_topics: list, limit=15) -> list:
    all_kw = keywords_en + keywords_ar + legal_topics
    if not all_kw:
        return []

    kw_rows  = _run(KEYWORD_Q, keywords=all_kw, limit=limit)
    sec_rows = _run(SECTION_Q, topics=legal_topics or keywords_en, limit=limit)

    # Merge: track max kw_hits for normalisation
    merged = {}
    for r in kw_rows + sec_rows:
        aid = r['id']
        if aid not in merged or r['kw_hits'] > merged[aid]['kw_hits']:
            merged[aid] = r

    max_hits = max((v['kw_hits'] for v in merged.values()), default=1)
    return [{
        'id': v['id'], 'number': v['number'],
        'english': v['english'], 'arabic': v['arabic'],
        'kw_score': v['kw_hits'] / max_hits,
        'sem_score': 0.0,
        'source': 'keyword'
    } for v in merged.values()]

# ── 3. Semantic vector search ──────────────────────────────────────────────
VECTOR_Q = """
CALL db.index.vector.queryNodes('article_embedding', $topK, $vec)
YIELD node AS a, score
RETURN a.id AS id, a.number AS number,
       a.english AS english, a.arabic AS arabic,
       score AS sem_score
"""

def fetch_by_semantic(query_text: str, top_k=15) -> list:
    vec = embed(query_text)
    rows = _run(VECTOR_Q, vec=vec, topK=top_k)
    return [{
        'id': r['id'], 'number': r['number'],
        'english': r['english'], 'arabic': r['arabic'],
        'kw_score': 0.0,
        'sem_score': float(r['sem_score']),
        'source': 'semantic'
    } for r in rows]

print('Retrieval functions defined.')

## Step 7 — Reranker

Merges all retrieved candidates and computes a **combined score**:

```
score = 0.55 × semantic_score + 0.35 × keyword_score + 0.10 × direct_bonus
```

Direct article matches (when user mentions article numbers) always rank first.

In [ ]:
def rerank(direct: list, keyword: list, semantic: list, top_k=5) -> list:
    pool = {}  # id → record

    def add(records, field, weight):
        for r in records:
            aid = r['id']
            if aid not in pool:
                pool[aid] = {
                    'id':      r['id'],
                    'number':  r['number'],
                    'english': r['english'],
                    'arabic':  r['arabic'],
                    'score':   0.0,
                    'source':  r.get('source', ''),
                }
            pool[aid]['score'] += r.get(field, 0.0) * weight

    # Direct hits get a +1 bonus (after normalization still dominates)
    for r in direct:
        r['kw_score']  = 1.0
        r['sem_score'] = 1.0

    add(direct,   'sem_score', 0.55)
    add(direct,   'kw_score',  0.35)
    add(keyword,  'kw_score',  0.35)
    add(keyword,  'sem_score', 0.55)
    add(semantic, 'sem_score', 0.55)
    add(semantic, 'kw_score',  0.35)

    # Extra bonus for direct article mentions
    direct_ids = {r['id'] for r in direct}
    for aid in direct_ids:
        if aid in pool:
            pool[aid]['score'] += 0.10

    ranked = sorted(pool.values(), key=lambda x: x['score'], reverse=True)
    return ranked[:top_k]

print('Reranker defined.')

## Step 8 — Answer Generator (Qwen3:4b with thinking)

Qwen3 is a **thinking model** — it reasons internally before answering.  
We let it think for legal questions to get more accurate answers.

In [ ]:
ANSWER_SYSTEM = """\
You are a highly knowledgeable legal assistant specializing in Egyptian Civil Law.

You will be given:
1. A question from the user (English or Arabic)
2. A set of relevant law articles retrieved from the database, each with Arabic and English text

Your task:
- Answer the question accurately based ONLY on the provided articles
- Cite the article number(s) you relied on (e.g., "According to Article 7...")
- If the articles do not contain enough information, say so clearly
- If the question is in Arabic, answer in Arabic; if in English, answer in English
- Be precise and professional — this is a legal context
"""

def format_articles(articles: list) -> str:
    parts = []
    for art in articles:
        parts.append(
            f"--- {art['id']} (score={art['score']:.3f}) ---\n"
            f"[English]\n{art['english']}\n\n"
            f"[Arabic]\n{art['arabic']}"
        )
    return '\n\n'.join(parts)

def generate_answer(question: str, articles: list) -> str:
    if not articles:
        return 'No relevant articles were found in the database for this question.'

    context = format_articles(articles)
    user_msg = f"RETRIEVED LAW ARTICLES:\n\n{context}\n\n---\n\nQUESTION: {question}"

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': ANSWER_SYSTEM},
            {'role': 'user',   'content': user_msg},
        ],
        options={'temperature': 0.3},
    )
    raw = response['message']['content']
    # Strip the <think>...</think> block from visible output (keep the answer only)
    answer = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    return answer

print('Answer generator defined.')

## Step 9 — Full RAG Pipeline

In [ ]:
def rag(question: str, top_k: int = 5, verbose: bool = True) -> str:
    """
    End-to-end RAG:
      1. Extract metadata from question
      2. Hybrid retrieval (keyword + section + semantic)
      3. Rerank candidates
      4. Generate answer with Qwen3
    """
    sep = '─' * 60

    # ── Stage 1: Metadata extraction ──────────────────────────────
    if verbose:
        print(f'{sep}\n📋 Extracting metadata...')
    meta = extract_metadata(question)
    if verbose:
        print(f"  Keywords EN : {meta.get('keywords_en', [])}")
        print(f"  Keywords AR : {meta.get('keywords_ar', [])}")
        print(f"  Legal topics: {meta.get('legal_topics', [])}")
        print(f"  Articles    : {meta.get('article_numbers', [])}")
        print(f"  Search query: {meta.get('search_query', '')}")

    # ── Stage 2: Hybrid retrieval ──────────────────────────────────
    if verbose:
        print(f'{sep}\n🔍 Retrieving from Neo4j...')

    direct   = fetch_by_number(meta.get('article_numbers', []))
    keyword  = fetch_by_keywords(
                   meta.get('keywords_en', []),
                   meta.get('keywords_ar', []),
                   meta.get('legal_topics', [])
               )
    semantic = fetch_by_semantic(meta.get('search_query', question))

    if verbose:
        print(f'  Direct hits   : {len(direct)}')
        print(f'  Keyword hits  : {len(keyword)}')
        print(f'  Semantic hits : {len(semantic)}')

    # ── Stage 3: Rerank ───────────────────────────────────────────
    top = rerank(direct, keyword, semantic, top_k=top_k)

    if verbose:
        print(f'{sep}\n🏆 Top {len(top)} articles after reranking:')
        for art in top:
            preview = art['english'][:80].replace('\n', ' ')
            print(f"  {art['id']:12}  score={art['score']:.3f}  {preview}...")

    # ── Stage 4: Answer ───────────────────────────────────────────
    if verbose:
        print(f'{sep}\n💬 Generating answer with Qwen3:4b...')
    answer = generate_answer(question, top)

    if verbose:
        print(f'{sep}')

    return answer

print('RAG pipeline ready.')

## Step 10 — Test Queries

In [ ]:
# ── English question ──────────────────────────────────────────────────────
question_en = "What are the conditions under which exercising a right becomes unlawful?"

answer = rag(question_en, top_k=5, verbose=True)
print('\n📖 ANSWER:\n')
print(answer)

In [ ]:
# ── Arabic question ───────────────────────────────────────────────────────
question_ar = "ما هي القواعد المتعلقة بالتقادم وانقضاء المدد الزمنية في القانون المدني؟"

answer = rag(question_ar, top_k=5, verbose=True)
print('\n📖 ANSWER:\n')
print(answer)

In [ ]:
# ── Question mentioning specific article numbers ───────────────────────────
question_art = "Explain Article 5 and how it relates to Article 4"

answer = rag(question_art, top_k=5, verbose=True)
print('\n📖 ANSWER:\n')
print(answer)

## Step 11 — Interactive Q&A Session

Run this cell and type your questions. Type `exit` to stop.

In [ ]:
print('=' * 60)
print(' Egyptian Civil Law — Interactive Q&A')
print(' Ask in English or Arabic. Type "exit" to quit.')
print('=' * 60)

while True:
    print()
    try:
        question = input('❓ Your question: ').strip()
    except EOFError:
        break

    if not question or question.lower() == 'exit':
        print('Goodbye!')
        break

    answer = rag(question, top_k=5, verbose=True)
    print('\n📖 ANSWER:\n')
    print(answer)
    print()